# Task 3 — Feature engineering

**Problem-statement steps:** *Create variables to represent the total number of children, age, and total spending.* and *Derive the total purchases from the number of transactions across the three channels.*

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

# Locate the raw data whether the notebook is opened from its own folder
# (notebooks/) or from the project root.
import os
_CANDIDATES = ["marketing_data.csv", "../marketing_data.csv",
               os.path.join(os.path.dirname(os.getcwd()), "marketing_data.csv")]
DATA_PATH = next((p for p in _CANDIDATES if os.path.exists(p)), "marketing_data.csv")
print("Using data file:", DATA_PATH)

Using data file: ../marketing_data.csv


In [2]:
REFERENCE_YEAR = 2015  # data compiled just after the last enrolment (Jun 2014)
SPEND_COLS = ["MntWines", "MntFruits", "MntMeatProducts",
              "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
CHANNEL_COLS = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]

def load_and_prepare(path=DATA_PATH, engineer=True):
    """Load -> fix dtypes -> clean categories -> impute income -> features."""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()                     # ' Income ' -> 'Income'
    if not pd.api.types.is_numeric_dtype(df["Income"]):     # '$84,835.00' -> float
        df["Income"] = (df["Income"].astype("string")
                        .str.replace(r"[\$,]", "", regex=True).str.strip().astype(float))
    df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%m/%d/%y")
    df["Marital_Status"] = df["Marital_Status"].replace(
        {"Alone": "Single", "YOLO": "Single", "Absurd": "Single"})
    df["Income"] = df.groupby(["Education", "Marital_Status"])["Income"].transform(
        lambda s: s.fillna(s.median()))
    df["Income"] = df["Income"].fillna(df["Income"].median())
    if engineer:
        df["Kids"] = df["Kidhome"] + df["Teenhome"]
        df["Age"] = REFERENCE_YEAR - df["Year_Birth"]
        df["Total_Spending"] = df[SPEND_COLS].sum(axis=1)
        df["Total_Purchases"] = df[CHANNEL_COLS].sum(axis=1)
        df["Total_Accepted_Cmp"] = df[["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3",
                                       "AcceptedCmp4", "AcceptedCmp5"]].sum(axis=1)
        df["Has_Child"] = (df["Kids"] > 0).astype(int)
        df["Is_US"] = (df["Country"] == "US").astype(int)
    return df

def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

def treat_outliers(df):
    """Drop impossible ages and winsorise heavy-tailed continuous columns."""
    df = df[df["Age"] <= 100].copy()
    for col in ["Income", "Total_Spending", "Total_Purchases", "NumWebVisitsMonth"]:
        lo, hi = iqr_bounds(df[col])
        df[col] = df[col].clip(lo, hi)
    return df

## 1. Start from the cleaned + imputed data (Tasks 1–2)

In [3]:
df = load_and_prepare(engineer=False)
print("Shape:", df.shape)

Shape: (2240, 28)


## 2. Create the derived variables
* `Kids` = `Kidhome` + `Teenhome`
* `Age` = 2015 − `Year_Birth`  (fixed reference year for reproducibility)
* `Total_Spending` = sum of the six `Mnt*` product columns
* `Total_Purchases` = Web + Catalog + Store purchases (the three channels)

In [4]:
df["Kids"] = df["Kidhome"] + df["Teenhome"]
df["Age"] = REFERENCE_YEAR - df["Year_Birth"]
df["Total_Spending"] = df[SPEND_COLS].sum(axis=1)
df["Total_Purchases"] = df[CHANNEL_COLS].sum(axis=1)
df[["Year_Birth", "Age", "Kidhome", "Teenhome", "Kids",
    "Total_Spending", "Total_Purchases"]].head(8)

,Year_Birth,Age,Kidhome,Teenhome,Kids,Total_Spending,Total_Purchases
0,1970,45,0,0,0,1190,14
1,1961,54,0,0,0,577,17
2,1958,57,0,1,1,251,10
3,1967,48,1,1,2,11,3
4,1989,26,1,0,1,91,6
5,1958,57,0,0,0,1192,16
6,1954,61,0,0,0,1215,27
7,1967,48,0,1,1,96,6


## 3. Summary of the new features

In [5]:
df[["Age", "Kids", "Total_Spending", "Total_Purchases"]].describe().round(2)

,Age,Kids,Total_Spending,Total_Purchases
count,2240.00,2240.00,2240.00,2240.00
mean,46.19,0.95,605.80,12.54
std,11.98,0.75,602.25,7.21
min,19.00,0.00,5.00,0.00
25%,38.00,0.00,68.75,6.00
50%,45.00,1.00,396.00,12.00
75%,56.00,1.00,1045.50,18.00
max,122.00,3.00,2525.00,32.00


In [6]:
print("Max Age =", int(df["Age"].max()), "(implausible birth years -> outliers for Task 4)")
df["Kids"].value_counts().sort_index()

Max Age = 122 (implausible birth years -> outliers for Task 4)


Kids
0     638
1    1128
2     421
3      53
Name: count, dtype: int64

### Conclusion
Four core engineered variables are ready for the distribution, correlation and hypothesis-testing tasks. `Age` = 122 flags the impossible birth years to treat in **Task 4**.